In [ ]:
from pathlib import Path
import sys
import ast
import importlib

REPO = Path("/workspace/ProtoSAM-repro")
PASTA = REPO / "exp" / "eft_validacao_polipos_v1"

assert Path(sys.executable).resolve() == Path(
    "/workspace/miniforge3/envs/protosam/bin/python"
).resolve(), f"Kernel incorreto: {sys.executable}"

FONTE = PASTA / "validar_polipos.py"
assert FONTE.is_file(), f"Avaliador anterior não encontrado: {FONTE}"

codigo = FONTE.read_text()

def trocar(antes, depois):
    global codigo
    assert codigo.count(antes) == 1, (
        f"Avaliador diferente do esperado. Trecho: {antes}"
    )
    codigo = codigo.replace(antes, depois, 1)

trocar(
    "def executar(piloto, retomar=None):",
    "def executar(piloto, passo, baseline, retomar=None):",
)

trocar(
    "    assert resumo_piloto['atualizacoes'] == 200",
    "    assert variante == 'tiny' and resumo_piloto['atualizacoes'] == 2000\n"
    "    assert passo in (1000, 1500, 2000)",
)

trocar(
    "    ckpt_eft = piloto / 'checkpoint_latest.pt'",
    "    ckpt_eft = piloto / f'checkpoint_{passo:04d}.pt'\n"
    "    assert ckpt_eft.is_file(), ckpt_eft",
)

trocar(
    "assert salvo['passo'] == 200 and salvo['config'] == cfg_piloto",
    "assert salvo['passo'] == passo and salvo['config'] == cfg_piloto",
)

trocar(
    "'finalidade':'Validação anotada antes/depois do piloto; não é teste final',",
    "'finalidade':'Validação anotada de checkpoints EFT; não é teste final',\n"
    "        'passo':passo, 'checkpoint_eft':str(ckpt_eft),",
)

trocar(
    "    pasta = Path(retomar).resolve() if retomar else "
    "REPO/'reproducao'/f'eft_{variante}_validacao_polipos_{stamp()}'",
    '''
    baseline = Path(baseline).resolve()
    ref = ler(baseline / 'config.json')

    for chave in (
        'variante', 'episodios', 'suporte', 'resolucao_encoder', 'canvas',
        'estagio', 'norm_antes', 'sam', 'selecao_sam', 'alp_inferencia',
        'grade_features', 'kernel_configurado', 'checkpoint_original_sha256',
        'checkpoint_sam_sha256', 'torch', 'cuda', 'versoes',
        'suporte_tensor_sha256', 'cudnn', 'tf32_matmul', 'tf32_cudnn',
        'gpu_driver',
    ):
        assert ref[chave] == config[chave], f'Baseline incompatível: {chave}'

    for fonte, digest in config['fontes_sha256'].items():
        if fonte in ref['fontes_sha256']:
            assert ref['fontes_sha256'][fonte] == digest, fonte

    assert ler(baseline / 'resumo.json')['n'] == 145
    config['baseline_reutilizada'] = str(baseline)
    config['baseline_config_sha256'] = sha(baseline / 'config.json')

    pasta = (
        Path(retomar).resolve()
        if retomar
        else REPO/'reproducao'/f'eft_tiny_passo_{passo}_validacao_{stamp()}'
    )
''',
)

trocar(
    "    modelo = construir_672(variante)",
    '''
    if not (pasta / 'antes').exists():
        temporario = pasta / f'antes_tmp_{stamp()}'
        shutil.copytree(baseline / 'antes', temporario)
        temporario.rename(pasta / 'antes')

    print(
        'Baseline reutilizada; suas máscaras serão conferidas por hash.',
        flush=True,
    )
    modelo = construir_672(variante)
''',
)

trocar(
    "resumo={'variante':variante,'n':145,",
    "resumo={'variante':variante,'passo':passo,'n':145,",
)

ast.parse(codigo)
DESTINO = PASTA / "validar_marcos.py"
DESTINO.write_text(codigo)

for caminho in (REPO, PASTA):
    if str(caminho) in sys.path:
        sys.path.remove(str(caminho))
    sys.path.insert(0, str(caminho))

importlib.invalidate_caches()
sys.modules.pop("validar_marcos", None)

from validar_marcos import executar, ler, salvar

print("Avaliador preparado:", DESTINO)
print("Nenhuma avaliação iniciada.")

In [ ]:
from pathlib import Path
import os

REPO = Path("/workspace/ProtoSAM-repro")
os.chdir(REPO)

print("Pasta de trabalho:", Path.cwd())

for relativo in (
    "data/PolypDataset/TrainDataset/images",
    "data/PolypDataset/TrainDataset/masks",
):
    caminho = REPO / relativo
    assert caminho.is_dir(), f"Pasta ausente ou link quebrado: {caminho}"

print("Caminhos de treino encontrados. Execute novamente a célula 2.")

In [ ]:
import json

TREINO = REPO / (
    "reproducao/eft_tiny_2000_v2_20261011T204725195685Z"
)

BASELINE = REPO / (
    "reproducao/eft_tiny_validacao_polipos_20261011T172038284160Z"
)

PASSOS = (1000, 1500, 2000)

assert (TREINO / "resumo.json").is_file()
assert (BASELINE / "resumo.json").is_file()

for passo in PASSOS:
    checkpoint = TREINO / f"checkpoint_{passo:04d}.pt"
    assert checkpoint.is_file(), f"Checkpoint ausente: {checkpoint}"

CONTROLE = REPO / "reproducao" / (
    "avaliacao_marcos_tiny_20261011T204725195685Z.json"
)

estado = (
    ler(CONTROLE)
    if CONTROLE.exists()
    else {"treino": str(TREINO), "avaliacoes": {}}
)
assert estado["treino"] == str(TREINO)

for passo in PASSOS:
    chave = str(passo)
    retomar = estado["avaliacoes"].get(chave)

    # Recupera também uma avaliação interrompida antes de retornar seu caminho.
    if retomar is None:
        candidatas = []

        for arquivo in (REPO / "reproducao").glob(
            f"eft_tiny_passo_{passo}_validacao_*/config.json"
        ):
            configuracao = ler(arquivo)
            if configuracao.get("piloto") == str(TREINO):
                candidatas.append(arquivo.parent)

        assert len(candidatas) <= 1, (
            f"Mais de uma avaliação encontrada para {passo}: {candidatas}"
        )

        if candidatas:
            retomar = str(candidatas[0])

    print(f"\n===== CHECKPOINT: {passo} PASSOS =====", flush=True)

    resultado = executar(
        piloto=TREINO,
        passo=passo,
        baseline=BASELINE,
        retomar=retomar,
    )

    estado["avaliacoes"][chave] = str(resultado)
    salvar(CONTROLE, estado)

# Consolida também os resultados já disponíveis de 0 e 200 passos.
piloto = ler(BASELINE / "resumo.json")

comparacao = [
    {
        "passo": 0,
        "Dice_ALP": piloto["ALP"]["Dice"]["antes"],
        "Dice_SAM": piloto["SAM"]["Dice"]["antes"],
    },
    {
        "passo": 200,
        "Dice_ALP": piloto["ALP"]["Dice"]["depois"],
        "Dice_SAM": piloto["SAM"]["Dice"]["depois"],
    },
]

for passo in PASSOS:
    resumo = ler(
        Path(estado["avaliacoes"][str(passo)]) / "resumo.json"
    )
    comparacao.append({
        "passo": passo,
        "Dice_ALP": resumo["ALP"]["Dice"]["depois"],
        "Dice_SAM": resumo["SAM"]["Dice"]["depois"],
    })

estado["comparacao"] = comparacao
salvar(CONTROLE, estado)

print("\nCOMPARAÇÃO FINAL")
print(json.dumps(comparacao, indent=2, ensure_ascii=False))
print("\nRegistro:", CONTROLE)

In [ ]:
from pathlib import Path
import sys
import os
import ast
import importlib

REPO = Path("/workspace/ProtoSAM-repro")
PASTA = REPO / "exp/eft_validacao_polipos_v1"

assert Path(sys.executable).resolve() == Path(
    "/workspace/miniforge3/envs/protosam/bin/python"
).resolve(), f"Kernel incorreto: {sys.executable}"

os.chdir(REPO)

fonte = PASTA / "validar_marcos.py"
assert fonte.is_file(), f"Avaliador anterior não encontrado: {fonte}"
codigo = fonte.read_text()

alteracoes = {
    "resumo_piloto['atualizacoes'] == 2000":
        "resumo_piloto['atualizacoes'] == 5000",
    "assert passo in (1000, 1500, 2000)":
        "assert passo in (3000, 4000, 5000)",
}

for antes, depois in alteracoes.items():
    assert codigo.count(antes) == 1, f"Trecho diferente do esperado: {antes}"
    codigo = codigo.replace(antes, depois, 1)

ast.parse(codigo)

destino = PASTA / "validar_marcos_5000.py"
destino.write_text(codigo)

for caminho in (REPO, PASTA):
    if str(caminho) in sys.path:
        sys.path.remove(str(caminho))
    sys.path.insert(0, str(caminho))

importlib.invalidate_caches()
sys.modules.pop("validar_marcos_5000", None)

from validar_marcos_5000 import executar, ler, salvar

print("Avaliador preparado:", destino)
print("Nenhuma avaliação iniciada.")

In [ ]:
import json

TREINO = REPO / (
    "reproducao/eft_tiny_5000_20261011T224215171455Z"
)
BASELINE = REPO / (
    "reproducao/eft_tiny_validacao_polipos_20261011T172038284160Z"
)
CONTROLE_ANTERIOR = REPO / (
    "reproducao/avaliacao_marcos_tiny_20261011T204725195685Z.json"
)
CONTROLE = REPO / (
    "reproducao/avaliacao_marcos_tiny_5000_20261011T224215171455Z.json"
)

PASSOS = (3000, 4000, 5000)

assert (TREINO / "resumo.json").is_file()
assert (BASELINE / "resumo.json").is_file()
assert CONTROLE_ANTERIOR.is_file()

for passo in PASSOS:
    checkpoint = TREINO / f"checkpoint_{passo:04d}.pt"
    assert checkpoint.is_file(), f"Checkpoint ausente: {checkpoint}"

estado = (
    ler(CONTROLE)
    if CONTROLE.exists()
    else {"treino": str(TREINO), "avaliacoes": {}}
)
assert estado["treino"] == str(TREINO)

for passo in PASSOS:
    chave = str(passo)
    retomar = estado["avaliacoes"].get(chave)

    if retomar is None:
        candidatas = []
        for arquivo in (REPO / "reproducao").glob(
            f"eft_tiny_passo_{passo}_validacao_*/config.json"
        ):
            configuracao = ler(arquivo)
            if configuracao.get("piloto") == str(TREINO):
                candidatas.append(arquivo.parent)

        assert len(candidatas) <= 1, (
            f"Mais de uma avaliação para {passo}: {candidatas}"
        )
        if candidatas:
            retomar = str(candidatas[0])

    print(f"\n===== CHECKPOINT: {passo} PASSOS =====", flush=True)

    resultado = executar(
        piloto=TREINO,
        passo=passo,
        baseline=BASELINE,
        retomar=retomar,
    )

    estado["avaliacoes"][chave] = str(resultado)
    salvar(CONTROLE, estado)

comparacao = list(ler(CONTROLE_ANTERIOR)["comparacao"])

for passo in PASSOS:
    resumo = ler(
        Path(estado["avaliacoes"][str(passo)]) / "resumo.json"
    )
    comparacao.append({
        "passo": passo,
        "Dice_ALP": resumo["ALP"]["Dice"]["depois"],
        "Dice_SAM": resumo["SAM"]["Dice"]["depois"],
    })

estado["comparacao"] = comparacao
salvar(CONTROLE, estado)

print("\nCOMPARAÇÃO FINAL")
print(json.dumps(comparacao, indent=2, ensure_ascii=False))
print("\nRegistro:", CONTROLE)